# SAM 3D Body 视频人体动作重建

本 Notebook 在 AMD ROCm / CUDA 环境下，使用 **ModelScope** 权重对视频做逐帧 3D 人体网格重建。

**流程**
1. 路径与依赖
2. 克隆 `sam-3d-body`；DINOv3 从 **Gitee** 镜像下载（`https://gitee.com/monkeycc/dinov3`）
3. ModelScope 下载 `facebook/sam-3d-body-dinov3`
4. 加载模型（可关 Detectron2，整图推理）
5. 视频逐帧推理并导出结果

> 建议先 `MAX_FRAMES=30` 测试，通过后再改为 `None` 处理全片。

## 0. 工作目录与路径配置

In [ ]:
import os
import sys

# ========== 路径配置（按你的机器修改） ==========
WORKSPACE = "/workspace"                                    # 工作根目录
REPO_DIR = os.path.join(WORKSPACE, "sam-3d-body")           # 官方推理代码仓库
DINOV3_DIR = os.path.join(WORKSPACE, "dinov3")              # DINOv3 本地代码（backbone）
DINOV3_GIT = "https://gitee.com/monkeycc/dinov3.git"        # 国内镜像，避免 GitHub 断连
VIDEO_PATH = os.path.join(WORKSPACE, "文艺甜1023281.mp4")   # 输入视频
OUTPUT_DIR = os.path.join(WORKSPACE, "output_video_results")  # 输出目录
CKPT_CACHE = os.path.join(REPO_DIR, "checkpoints")          # 权重缓存

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_CACHE, exist_ok=True)

print("WORKSPACE:", WORKSPACE)
print("REPO_DIR:", REPO_DIR)
print("DINOV3_GIT:", DINOV3_GIT)
print("VIDEO_PATH:", VIDEO_PATH)
print("视频存在:", os.path.exists(VIDEO_PATH))

## 1. 安装依赖

`xtcocotools` 易编译失败，本流程不安装。

In [ ]:
!pip install -U pip setuptools wheel numpy -q

!pip install -q \
    pytorch-lightning pyrender opencv-python yacs scikit-image \
    einops timm dill pandas rich hydra-core pyrootutils roma \
    joblib seaborn loguru optree fvcore pycocotools huggingface_hub \
    modelscope networkx==3.2.1 tqdm braceexpand webdataset jsonlines \
    appdirs ffmpeg-python

print("基础依赖安装完成")

In [ ]:
# Detectron2 可选：仅在需要自动人体框检测时安装
# 若 pkg_resources / Detectron2 报错，后面设 USE_DETECTOR=False 即可
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps

try:
    import pkg_resources
    print("pkg_resources OK")
except Exception as e:
    print("pkg_resources 异常，将使用无检测器模式:", e)

## 2. 克隆仓库；DINOv3 使用 Gitee 镜像

In [ ]:
import os

# 官方 SAM 3D Body 代码（含 notebook.utils / tools）
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/facebookresearch/sam-3d-body.git {REPO_DIR}
else:
    print("sam-3d-body 已存在:", REPO_DIR)

# DINOv3 backbone 代码：使用 Gitee 镜像，避免 GitHub RemoteDisconnected
if not os.path.exists(DINOV3_DIR):
    !git clone {DINOV3_GIT} {DINOV3_DIR}
else:
    print("dinov3 已存在:", DINOV3_DIR)

%cd {REPO_DIR}
print("当前目录:", os.getcwd())
print("notebook/utils.py:", os.path.exists("notebook/utils.py"))
print("tools/vis_utils.py:", os.path.exists("tools/vis_utils.py"))

In [ ]:
import torch

# SAM 3D Body 内部会执行:
#   torch.hub.load("facebookresearch/dinov3", name, source="github", ...)
# 在国内常因网络失败。这里劫持 hub.load，改为从本地 Gitee 克隆目录加载。
_original_hub_load = torch.hub.load

def local_hub_load(repo_or_dir, model, source="github", *args, **kwargs):
    if repo_or_dir == "facebookresearch/dinov3":
        print(f"[hub] 使用本地 dinov3（Gitee）: {DINOV3_DIR}")
        return _original_hub_load(DINOV3_DIR, model, source="local", *args, **kwargs)
    return _original_hub_load(repo_or_dir, model, source=source, *args, **kwargs)

torch.hub.load = local_hub_load
print("已设置本地 DINOv3 加载（来源: gitee.com/monkeycc/dinov3）")

## 3. 从 ModelScope 下载权重

In [ ]:
from modelscope import snapshot_download

# 国内从 ModelScope 拉权重，不走 HuggingFace 鉴权
model_dir = snapshot_download(
    "facebook/sam-3d-body-dinov3",
    cache_dir=CKPT_CACHE,
)
print("模型目录:", model_dir)

# 不同 cache 结构下文件层级可能不同，递归查找
checkpoint_path = None  # 主权重 model.ckpt
mhr_path = None         # Momentum Human Rig 资产 mhr_model.pt
for root, dirs, files in os.walk(model_dir):
    for f in files:
        if f == "model.ckpt":
            checkpoint_path = os.path.join(root, f)
        if f == "mhr_model.pt":
            mhr_path = os.path.join(root, f)

print("checkpoint_path:", checkpoint_path)
print("mhr_path:", mhr_path)

assert checkpoint_path and os.path.exists(checkpoint_path), "未找到 model.ckpt"
assert mhr_path and os.path.exists(mhr_path), "未找到 mhr_model.pt"

## 4. 加载 SAM 3D Body

- `USE_DETECTOR=True`：ViTDet 自动检框（需 Detectron2 正常）
- `USE_DETECTOR=False`：整图推理（单人视频更省事）

In [ ]:
import sys
sys.path.insert(0, os.getcwd())  # 保证能 import sam_3d_body / tools

import torch
from sam_3d_body import load_sam_3d_body, SAM3DBodyEstimator
from tools.vis_utils import visualize_sample_together

# False = 不加载 Detectron2，对整张图重建（推荐先这样跑通）
USE_DETECTOR = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用设备:", device)

# 注意：不要用 setup_sam_3d_body(checkpoint_path=...)
# 该函数只认 hf_repo_id；本地权重必须走 load_sam_3d_body
model, model_cfg = load_sam_3d_body(
    checkpoint_path=checkpoint_path,
    device=device,
    mhr_path=mhr_path,
)

human_detector = None
if USE_DETECTOR:
    try:
        from tools.build_detector import HumanDetector
        print("加载人体检测器 ViTDet...")
        human_detector = HumanDetector(name="vitdet", device=device)
    except Exception as e:
        print("检测器加载失败，回退整图模式:", e)
        human_detector = None

estimator = SAM3DBodyEstimator(
    sam_3d_body_model=model,
    model_cfg=model_cfg,
    human_detector=human_detector,
)
print("模型加载完成 | 检测器:", "开启" if human_detector else "关闭（整图）")

## 5. 视频逐帧推理（带详细注释）

In [ ]:
"""
SAM 3D Body 视频转 3D 人体网格
--------------------------------
对输入 mp4 按设定步长抽帧，每帧做 3D 人体重建，
把 mesh 叠加回画面，写出结果视频。
"""
import cv2
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

# -------------------- 推理超参数 --------------------
# 每隔 N 帧处理 1 帧：1=全帧最准最慢；3/5 适合先快速看效果
PROCESS_EVERY_N_FRAMES = 3

# 最多处理多少帧；测试时用 30，全片改为 None
MAX_FRAMES = 30

# 是否写出带可视化的 mp4
SAVE_VIDEO = True

# 是否在 Notebook 里实时弹图（会拖慢一点）
SHOW_PREVIEW = True

# -------------------- 打开输入视频 --------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise ValueError(f"无法打开视频: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)                      # 原视频帧率
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"视频: {width}x{height}, {fps:.1f} FPS, 总帧数 {total_frames}")

# -------------------- 准备输出视频 --------------------
out_writer = None
out_video_path = os.path.join(OUTPUT_DIR, "文艺甜_SAM3D_result.mp4")
if SAVE_VIDEO:
    # mp4v 兼容性较好；输出帧率按抽帧比例缩放，时长大致对齐原片
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_fps = max(fps / PROCESS_EVERY_N_FRAMES, 1.0)
    out_writer = cv2.VideoWriter(out_video_path, fourcc, out_fps, (width, height))

# -------------------- 主循环 --------------------
frame_idx = 0          # 当前读到的原始帧编号
processed_count = 0    # 实际做过推理的帧数
results_list = []      # 记录“检测到人”的帧，便于统计

limit = total_frames if MAX_FRAMES is None else min(MAX_FRAMES, total_frames)
print("开始处理...")
pbar = tqdm(total=limit)

while True:
    ret, frame_bgr = cap.read()   # OpenCV 默认 BGR
    if not ret:
        break  # 读完或读失败

    # 达到测试上限则停
    if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
        break

    # 跳帧：只处理 frame_idx % N == 0 的帧
    if frame_idx % PROCESS_EVERY_N_FRAMES != 0:
        frame_idx += 1
        pbar.update(1)
        continue

    # 模型期望 RGB
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    outputs = []  # 每人一个 dict：顶点、2D/3D 关键点、相机等

    try:
        # 核心推理：有检测器时先检框再重建；无检测器则整图输入
        outputs = estimator.process_one_image(frame_rgb)

        if outputs:
            # 把 3D mesh 渲染叠回原图（BGR）
            vis_img = visualize_sample_together(frame_bgr, outputs, estimator.faces)
            vis_img = vis_img.astype(np.uint8)
            results_list.append({
                "frame_idx": frame_idx,
                "num_people": len(outputs),
            })
        else:
            # 未检测到人：原图画提示文字
            vis_img = frame_bgr.copy()
            cv2.putText(
                vis_img, "No person detected", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 2,
            )
    except Exception as e:
        # 单帧失败不中断整段视频
        print(f"\n第 {frame_idx} 帧出错: {e}")
        vis_img = frame_bgr.copy()

    # 写入结果视频
    if out_writer is not None:
        out_writer.write(vis_img)

    # Notebook 预览（每处理 3 次更新一次，减少刷屏）
    if SHOW_PREVIEW and processed_count % 3 == 0:
        clear_output(wait=True)
        plt.figure(figsize=(12, 7))
        plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
        plt.title(f"Frame {frame_idx} | 人数 {len(outputs) if outputs else 0}")
        plt.axis("off")
        plt.show()

    processed_count += 1
    frame_idx += 1
    pbar.update(1)

# -------------------- 收尾 --------------------
cap.release()
if out_writer is not None:
    out_writer.release()
    print("输出视频:", out_video_path)

pbar.close()
print(f"完成：推理 {processed_count} 帧，其中有人物 {len(results_list)} 帧")

## 6. 全片处理

测试通过后，把第 5 节中的参数改为：

```python
MAX_FRAMES = None
SHOW_PREVIEW = False          # 全片时建议关预览提速
PROCESS_EVERY_N_FRAMES = 2    # 可按显存/速度再调
```

然后重新运行第 5 节单元格。

## 常见问题

| 现象 | 处理 |
|------|------|
| `No module named 'notebook.utils'` | 先 `%cd` 到 `sam-3d-body` |
| `setup_sam_3d_body() unexpected keyword` | 本地权重用 `load_sam_3d_body`，不要传 `checkpoint_path` 给 setup |
| GitHub / hub `RemoteDisconnected` | DINOv3 用 Gitee：`https://gitee.com/monkeycc/dinov3` + 本地 hub patch |
| `pkg_resources` / Detectron2 | `pip install -U setuptools`，或 `USE_DETECTOR=False` |
| 权重下载 | ModelScope：`facebook/sam-3d-body-dinov3` |